1.将输入

In [ ]:
from datetime import datetime
import pandas as pd
from pathlib import Path
from rdkit import Chem

from summit.domain import CategoricalVariable, ContinuousVariable
from summit import Domain
from summit.utils.dataset import DataSet

from condition_opt.EDBOplus.edbo import newEDBO
from condition_opt.utils import descClass, get_reaction_space, get_target_value, canonicalize_smiles


bayesian_opt_round = 'test' # selecting bayesian optimization round
data_path = Path("condition_opt")  # Ensure data_path is a Path object
batch_size = 10 # number of experiments to suggest

exp_datapath = data_path / Path(f"opt_round_{bayesian_opt_round}/manual_conditions_new.csv")

# generate reaction space
reagent_types = ["tempo", "additive", "ratio", "solvent", "volume"]
desc_class = descClass(Path("descriptors"))
domain = Domain()
domain = get_reaction_space(domain, desc_class, reagent_types=reagent_types)

# generate target columns
domain = get_target_value(domain)

new_batch_id = 0
# proceed EDBO
print("Beginning EDBO initialization...")
edbo = newEDBO(domain=domain, seed=1216, init_sampling_method="LHS")
print("Beginning New reaction suggestion...")
edbo_restults = edbo.suggest_experiments(batch_size=batch_size) # generate new experiments

# save results
formatted_date = datetime.now().strftime("%Y%m%d")  # get data with format 'yyyymmdd'
edbo_restults.to_csv(data_path / Path(f"opt_round_{bayesian_opt_round}/edbo-results_batch-{new_batch_id}_{formatted_date}.csv"))

bayesian_opt_round = 'test'每次改成对应序号

In [2]:
from datetime import datetime
import pandas as pd
from pathlib import Path
from rdkit import Chem
from sklearn.preprocessing import StandardScaler

from summit.domain import CategoricalVariable, ContinuousVariable
from summit import Domain
from summit.utils.dataset import DataSet

from condition_opt.EDBOplus.edbo import newEDBO
from condition_opt.utils import descClass, get_reaction_space, get_target_value, canonicalize_smiles


bayesian_opt_round = '2' # selecting bayesian optimization round
data_path = Path("condition_opt")  # Ensure data_path is a Path object
batch_size = 10 # number of experiments to suggest

exp_datapath = data_path / Path(f"opt_round_{bayesian_opt_round}/manual_conditions_new.csv")

# generate reaction space
reagent_types = ["tempo", "additive", "ratio", "solvent", "volume"]
desc_class = descClass(Path("descriptors"))
domain = Domain()
domain = get_reaction_space(domain, desc_class, reagent_types=reagent_types)

# generate target columns
domain = get_target_value(domain)

data_df = pd.read_csv(exp_datapath)
data_df = data_df[data_df["select_tag"] == True]
new_batch_id = data_df["batch_id"].max() + 1

# canonicalize smiles in data_df except for cobalt.
#for tp in reagent_types:
#    if tp != "cobalt":
#        data_df[f"{tp}_smiles"] = data_df[f"{tp}_smiles"].apply(canonicalize_smiles)

# process for done experments datas
done_dataset = data_df.iloc[:, 1:]
done_dataset.columns = reagent_types + ["yld", "select_tag"]

# check if all batch molecules in reaction space
desc_dict = desc_class.get_desc_df()

done_dataset.reset_index(inplace=True, drop=True)
for i in done_dataset.index:
    for tp in reagent_types:
        if done_dataset.loc[i, tp] not in desc_dict[tp].index:
            print(f"Invalid {tp} molecule: {done_dataset.loc[i, tp]}")
            done_dataset.loc[i, "select_tag"] = False

print(done_dataset["select_tag"].value_counts())
done_dataset = done_dataset[done_dataset["select_tag"]]
done_dataset = done_dataset.drop(columns=["select_tag"])

done_dataset.reset_index(inplace=True)
done_dataset = DataSet.from_df(done_dataset)

# proceed EDBO
print("Beginning EDBO initialization...")
edbo = newEDBO(domain=domain, seed=1216, init_sampling_method="LHS")
print("Beginning New reaction suggestion...")
edbo_restults = edbo.suggest_experiments_new(prev_res=done_dataset, batch_size=batch_size)

# save results
formatted_date = datetime.now().strftime("%Y%m%d")  # get data with format 'yyyymmdd'
edbo_restults.to_csv(data_path / Path(f"opt_round_{bayesian_opt_round}/edbo-results_batch-{new_batch_id}_{formatted_date}.csv"))

In [6]:
from datetime import datetime
import pandas as pd
from pathlib import Path
import logging
from rdkit import Chem

from summit.domain import Domain
from summit.utils.dataset import DataSet

from condition_opt.EDBOplus.edbo import newEDBO
from condition_opt.utils import descClass, get_target_value

# Configuration
CONFIG = {
    'bayesian_opt_round': '1',
    'data_path': Path("condition_opt"),
    'batch_size': 10,
    'reagent_types': ["tempo", "additive", "ratio", "solvent", "volume"],
    'seed': 42,
    'init_sampling_method': "LHS"
}

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def load_and_validate_data(exp_datapath, reagent_types, desc_dict):
    """Load and validate experimental data"""
    try:
        data_df = pd.read_csv(exp_datapath)
    except FileNotFoundError:
        logger.error(f"Data file not found: {exp_datapath}")
        raise
    
    # Filter selected experiments
    data_df = data_df[data_df["select_tag"] == True]
    logger.info(f"Loaded {len(data_df)} selected experiments")
    
    # Extract done experiments
    done_dataset = data_df.iloc[:, 1:]
    done_dataset.columns = reagent_types + ["yield", "select_tag"]
    
    # Validate against reaction space
    valid_mask = pd.Series(True, index=done_dataset.index)
    for tp in reagent_types:
        is_valid = done_dataset[tp].isin(desc_dict[tp].index)
        invalid_count = (~is_valid).sum()
        if invalid_count > 0:
            logger.warning(f"Found {invalid_count} invalid {tp} molecules")
            for idx in done_dataset[~is_valid].index:
                logger.warning(f"Invalid {tp} molecule: {done_dataset.loc[idx, tp]}")
        valid_mask = valid_mask & is_valid
    
    done_dataset.loc[~valid_mask, "select_tag"] = False
    done_dataset = done_dataset[done_dataset["select_tag"]]
    done_dataset = done_dataset.drop(columns=["select_tag"])
    
    logger.info(f"After validation: {len(done_dataset)} valid experiments")
    return data_df, done_dataset

def main():
    """Main execution function"""
    config = CONFIG
    
    # Setup paths
    exp_datapath = config['data_path'] / f"opt_round_{config['bayesian_opt_round']}/manual_conditions_new.csv"
    
    # Initialize domain and descriptors
    desc_class = descClass(Path("descriptors"))
    domain = Domain()
    domain = get_target_value(domain)
    desc_dict = desc_class.get_desc_df()
    
    # Load and validate data
    data_df, done_dataset = load_and_validate_data(exp_datapath, config['reagent_types'], desc_dict)
    new_batch_id = data_df["batch_id"].max() + 1
    
    # Convert to Summit DataSet
    done_dataset.reset_index(drop=True, inplace=True)
    done_dataset = DataSet.from_df(done_dataset)
    
    # Run Bayesian optimization
    logger.info("Beginning EDBO initialization...")
    edbo = newEDBO(domain=domain, seed=config['seed'], init_sampling_method=config['init_sampling_method'])
    
    logger.info("Beginning new reaction suggestion...")
    edbo_results = edbo.suggest_experiments_new(prev_res=done_dataset, batch_size=config['batch_size'])
    
    # Save results
    formatted_date = datetime.now().strftime("%Y%m%d")
    output_path = config['data_path'] / f"opt_round_{config['bayesian_opt_round']}/edbo-results_batch-{new_batch_id}_{formatted_date}.csv"
    edbo_results.to_csv(output_path)
    logger.info(f"Results saved to: {output_path}")

main()